# La Nación: exploración de la fuente (T2.3)

Lista las notas de Política con el sitemap diario (días 10 y 25 de cada mes desde el 10/12/2023), lee dos notas por día desde el JSON de Arc embebido en el HTML (`Fusion.globalContent`) y mide volumen, campos, notas en vivo y la relación entre la fecha de la URL y el día del sitemap.

No se usa la API del frontend (`/pf/api/`). Todas las requests pasan por `probe.py`: solo rutas permitidas por robots.txt, User-Agent identificable, 2 s entre requests y caché en `data/probe_cache/`. Con `PROBE_OFFLINE=1` el notebook corre solo desde la caché.

In [1]:
import re
from datetime import date, datetime, timedelta

import pandas as pd

import arc
import probe

p = probe.Probe()
LAST_MONTH = date(2026, 9, 1)  # último mes completo al explorar
pd.set_option("display.max_colwidth", 90)


def sitemap_url(day):
    return f"https://www.lanacion.com.ar/sitemap-articles-from{day:%Y-%m-%d}T00:00:00Z-to{day + timedelta(days=1):%Y-%m-%d}T00:00:00Z.xml"


NID = re.compile(r"-nid(\d{2})(\d{2})(\d{4})/?$")


def is_politica(url):
    return url.startswith("https://www.lanacion.com.ar/politica/")


def url_date(url):
    m = NID.search(url)
    return date(int(m[3]), int(m[2]), int(m[1])) if m else None

## 1. Sitemaps diarios

In [2]:
counts = arc.count_sitemaps(p, arc.sample_days(LAST_MONTH), sitemap_url, is_politica)
print("Sitemaps sin respuesta 200:", int((counts["status"] != 200).sum()))
counts.drop(columns="politica_urls")

Sitemaps sin respuesta 200: 0


,day,status,total_urls,politica
0,2023-12-10,200,337,76
1,2023-12-25,200,226,13
2,2024-01-10,200,430,29
3,2024-01-25,200,416,31
4,2024-02-10,200,314,16
...,...,...,...,...
63,2026-07-25,200,432,13
64,2026-08-10,200,497,31
65,2026-08-25,200,622,30
66,2026-09-10,200,672,26


## 2. Volumen estimado

Promedio de notas de Política de los dos días muestreados × días del mes. `spread_over_2x` marca meses donde un día tiene más del doble que el otro.

In [3]:
monthly = arc.monthly_estimate(counts)
display(monthly)
print("Total estimado desde el 10/12/2023 hasta el último mes completo:", monthly["estimated_notes"].sum())
print("Promedio diario de notas de Política:", round(counts["politica"].mean(), 1))

,month,sampled,estimated_notes,spread_over_2x
0,2023-12,"[76, 13]",979,True
1,2024-01,"[29, 31]",930,False
2,2024-02,"[16, 13]",420,False
3,2024-03,"[23, 30]",822,False
4,2024-04,"[44, 48]",1380,False
5,2024-05,"[26, 16]",651,False
6,2024-06,"[24, 27]",765,False
7,2024-07,"[28, 30]",899,False
8,2024-08,"[16, 24]",620,False
9,2024-09,"[32, 34]",990,False


Total estimado desde el 10/12/2023 hasta el último mes completo: 26602
Promedio diario de notas de Política: 26.1


## 3. Campos del esquema

In [4]:
notes = arc.read_notes(p, arc.pick_notes(counts))
print("Notas leídas:", len(notes), "| sin respuesta 200:", int((notes["status"] != 200).sum()))
arc.field_presence(notes)

Notas leídas: 136 | sin respuesta 200: 0


,column,source,present_share
0,id,hash of canonical_url (computed),1.000
1,medio,constant per outlet,NaN
2,url,"canonical_url (relative, prefixed with the site)",1.000
3,fecha_publicacion,"first_publish_date (UTC, converted to UTC-3)",1.000
4,titulo,headlines.basic,1.000
5,bajada,subheadlines.basic,0.985
6,cuerpo,"content_elements of type text, header and list (HTML)",0.985
7,autor,credits.by[].name,0.500
8,seccion,taxonomy.primary_section._id,1.000
9,tags,taxonomy.tags[].text,0.801


In [5]:
notes["cuerpo_words"].describe().round()

count     136.0
mean      808.0
std       342.0
min         0.0
25%       604.0
50%       724.0
75%       971.0
max      2220.0
Name: cuerpo_words, dtype: float64

## 4. Notas en vivo

In [6]:
notes["live"] = notes["live_signals"].str.len() > 0
print("Proporción de notas en vivo:", round(notes["live"].mean(), 3))
display(pd.Series([s for signals in notes["live_signals"] for s in signals]).value_counts().rename("señal"))
display(notes.groupby("live")["cuerpo_words"].describe().round())
display(notes["subtype"].value_counts(dropna=False).rename("subtype"))

Proporción de notas en vivo: 0.0


Series([], Name: señal, dtype: int64)

,count,mean,std,min,25%,50%,75%,max
live,,,,,,,,
False,136.0,808.0,342.0,0.0,604.0,724.0,971.0,2220.0


subtype
1    134
9      2
Name: subtype, dtype: int64

## 5. Fecha de la URL vs día del sitemap

Diferencia en días entre la fecha que trae cada URL de Política y el día del sitemap que la lista (todas las URLs de los sitemaps muestreados).

In [7]:
arc.url_day_offsets(counts, url_date)

offset_days
-338       1
-22        1
-10        1
-7         1
-5         1
-4         1
-3         5
-2        13
-1       381
 0      1371
Name: urls, dtype: int64

## 6. Paywall: `content_code`

Distribución de `content_restrictions.content_code` y largo del cuerpo por código. Si en la muestra no aparece ninguna nota con un código distinto de `comun` y `abierta` (los dos de acceso libre), se leen hasta 50 notas más elegidas al azar (semilla fija) entre las URLs de Política de los sitemaps muestreados. El sitemap no trae títulos, así que no se puede filtrar por tipo de nota antes de leerla.

In [8]:
def by_code(frame):
    return frame[frame["status"] == 200].groupby("content_code", dropna=False)["cuerpo_words"].describe().round()

FREE = {"comun", "abierta"}
display(by_code(notes))
if notes["content_code"].fillna("comun").isin(FREE).all():
    pool = sorted({u for urls in counts["politica_urls"] for u in urls} - set(notes["url"]))
    extra = pd.Series(pool).sample(min(50, len(pool)), random_state=7)
    url_day = {u: d for d, urls in zip(counts["day"], counts["politica_urls"]) for u in urls}
    extra_notes = arc.read_notes(p, [(url_day[u], u) for u in extra])
    print("Notas extra leídas:", len(extra_notes))
    display(by_code(extra_notes))
    notes_all = pd.concat([notes, extra_notes], ignore_index=True)
else:
    notes_all = notes
print("Notas con content_code distinto de 'comun' y 'abierta':",
      int((~notes_all["content_code"].fillna("comun").isin(FREE)).sum()), "de", len(notes_all))
display(notes_all["content_code"].value_counts(dropna=False).rename("notas"))

,count,mean,std,min,25%,50%,75%,max
content_code,,,,,,,,
abierta,3.0,575.0,20.0,559.0,564.0,568.0,582.0,597.0
comun,133.0,813.0,344.0,0.0,611.0,726.0,974.0,2220.0


Notas extra leídas: 50


,count,mean,std,min,25%,50%,75%,max
content_code,,,,,,,,
abierta,2.0,820.0,449.0,502.0,661.0,820.0,978.0,1137.0
comun,48.0,903.0,607.0,205.0,516.0,696.0,1052.0,3264.0


Notas con content_code distinto de 'comun' y 'abierta': 0 de 186


content_code
comun      181
abierta      5
Name: notas, dtype: int64

## 7. ¿El sitemap agrupa por publicación o por modificación?

Para 5 días repartidos en el período se leen también los sitemaps del día anterior y del siguiente, y se cuentan las URLs de Política que aparecen en más de uno. El `lastmod` de cada entrada se compara con `display_date` y `first_publish_date` de las notas leídas.

In [9]:
days = list(counts["day"].iloc[[0, 17, 34, 51, -1]])
display(arc.neighbour_overlap(p, days, sitemap_url, is_politica))

,day,politica,also_day_before,also_day_after
0,2023-12-10,76,0,0
1,2024-08-25,24,0,0
2,2025-05-10,13,0,0
3,2026-01-25,17,0,0
4,2026-09-25,18,0,0


In [10]:
lastmod = {loc: mod for day in counts["day"] for loc, mod in arc.sitemap_entries(p.get(sitemap_url(day)).text)}
ok = notes[notes["status"] == 200].copy()
ok["lastmod"] = ok["url"].map(lastmod)
same = lambda a, b: (pd.to_datetime(ok[a], utc=True, format="ISO8601") - pd.to_datetime(ok[b], utc=True, format="ISO8601")).abs() < pd.Timedelta(seconds=1)
pd.Series({"lastmod == display_date": same("lastmod", "display_date").mean(),
           "lastmod == first_publish_date": same("lastmod", "fecha_publicacion").mean()}).round(3)

lastmod == display_date          1.000
lastmod == first_publish_date    0.544
dtype: float64

## Ejemplos

Cinco notas de la muestra, con el título recortado (no se publica el cuerpo).

In [11]:
display(notes.sample(5, random_state=1)[["sitemap_day", "fecha_publicacion", "url", "cuerpo_words"]]
        .assign(titulo=notes["titulo"].str[:80]))
print("Requests de red en esta corrida:", p.network_requests)

,sitemap_day,fecha_publicacion,url,cuerpo_words,titulo
42,2024-10-25,2024-10-25T14:52:33.564Z,https://www.lanacion.com.ar/politica/el-gobierno-prohibio-los-cargos-hereditarios-en-e...,1073,El Gobierno prohibió los cargos hereditarios en el Estado
36,2024-09-10,2024-09-10T22:10:22.353Z,https://www.lanacion.com.ar/politica/luis-caputo-anuncio-que-los-municipios-ya-no-podr...,1053,Luis Caputo anunció que los municipios ya no podrán incluir impuestos y tasas en
93,2025-11-10,2025-11-10T14:40:31.381Z,https://www.lanacion.com.ar/politica/la-justicia-de-estados-unidos-le-dio-cinco-semana...,729,La Justicia de Estados Unidos le dio cinco semanas a Fred Machado para declarars
5,2024-01-10,2024-01-10T14:47:10.170Z,https://www.lanacion.com.ar/politica/la-embajada-de-china-expreso-su-malestar-por-supu...,619,La embajada de China expresó su malestar por supuestos contactos con Taiwán y Mo
65,2025-04-10,2025-04-09T13:26:40.834Z,https://www.lanacion.com.ar/politica/el-gobierno-asegura-que-el-viernes-el-fmi-va-a-ap...,731,"Tras la confirmación del acuerdo técnico del FMI, el Gobierno asegura que el vie"


Requests de red en esta corrida: 0
